# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [1]:
import polars as pl, duckdb

RAW = "../data/processed/data.parquet"
STAGE = "../data/processed/data_dedup.parquet"
CLEAN = "../data/processed/data_clean.parquet"

lf = pl.scan_parquet(RAW)
n_raw = lf.select(pl.len()).collect().item()
print(f"Baris mentah: {n_raw:,}")
duckdb.sql(f"SUMMARIZE SELECT * FROM '{RAW}'").show(max_width=250)

Baris mentah: 310,930


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────┬──────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────┬───────────────┬──────────┬───────────────────┬─────────┬─────────┬─────────┬────────┬─────────────────┐
│ column_name │ column_type │                           min                            │                           max                           │ approx_unique │   avg    │        std        │   q25   │   q50   │   q75   │ count  │ null_percentage │
│   varchar   │   varchar   │                         varchar                          │                         varchar                         │     int64     │ varchar  │      varchar      │ varchar │ varchar │ varchar │ int64  │  decimal(9,2)   │
├─────────────┼─────────────┼──────────────────────────────────────────────────────────┼─────────────────────────────────────────────────────────┼───────────────┼──────────┼───────────────────┼─────────┼─────────┼─────────┼────────┼───────────────

## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [2]:
MIN_CHAR = 185   # panjang minimum dokumen; sesuaikan dengan hasil profiling di notebook 01

miss = lf.select(
    pl.len().alias("total"),
    pl.col("text").is_null().sum().alias("null"),
    (pl.col("text").str.strip_chars() == "").sum().alias("kosong"),
    (pl.col("text").str.len_chars() < MIN_CHAR).sum().alias("lebih_pendek_dari_min"),
).collect()
miss = miss.with_columns([(pl.col(c) / pl.col("total") * 100).round(3).alias(f"{c}_%")
                          for c in ["null", "kosong", "lebih_pendek_dari_min"]])
miss

total,null,kosong,lebih_pendek_dari_min,null_%,kosong_%,lebih_pendek_dari_min_%
u32,u32,u32,u32,f64,f64,f64
310930,0,0,24,0.0,0.0,0.008


## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [3]:
# Teks adalah isi utama dokumen, jadi null/kosong/terlalu pendek tidak diimputasi, barisnya dibuang.
s0 = lf.with_columns(pl.col("text").str.replace_all(r"\s+", " ").str.strip_chars())
s1 = s0.filter(pl.col("text").is_not_null() & (pl.col("text") != ""))
s2 = s1.filter(pl.col("text").str.len_chars() >= MIN_CHAR)

log = {"mentah": n_raw}
log["setelah buang null/kosong"] = s1.select(pl.len()).collect().item()
log["setelah panjang minimum"] = s2.select(pl.len()).collect().item()
for k, v in log.items():
    print(f"{k:28s}: {v:,}")

mentah                      : 310,930
setelah buang null/kosong   : 310,930
setelah panjang minimum     : 310,906


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [4]:
s2 = s2.with_columns(pl.col("text").str.len_chars().alias("n_chars"))

q = s2.select(pl.col("n_chars").quantile(0.25).alias("q1"),
              pl.col("n_chars").quantile(0.75).alias("q3")).collect()
q1, q3 = q["q1"][0], q["q3"][0]
iqr = q3 - q1
batas_iqr15 = q3 + 1.5 * iqr      # outlier standar
batas_atas = q3 + 3 * iqr         # outlier ekstrem (dipakai untuk penandaan)
print(f"Q1={q1:,.0f}, Q3={q3:,.0f}, IQR={iqr:,.0f}")

cnt = s2.select(
    (pl.col("n_chars") > batas_iqr15).sum().alias("outlier_1.5xIQR"),
    (pl.col("n_chars") > batas_atas).sum().alias("outlier_ekstrem_3xIQR"),
).collect()
print(cnt)

# Teks tidak bisa di-clip (windsorization), dan dokumen panjang (mis. jurnal) valid,
# jadi outlier ditandai di kolom boolean, tidak dibuang.
s_flag = s2.with_columns((pl.col("n_chars") > batas_atas).alias("is_outlier_len"))

Q1=1,243, Q3=2,432, IQR=1,189
shape: (1, 2)
┌─────────────────┬───────────────────────┐
│ outlier_1.5xIQR ┆ outlier_ekstrem_3xIQR │
│ ---             ┆ ---                   │
│ u32             ┆ u32                   │
╞═════════════════╪═══════════════════════╡
│ 17202           ┆ 6781                  │
└─────────────────┴───────────────────────┘


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [5]:
# Kunci duplikat = isi teks (dataset tidak punya kolom ID unik; doc_id hanya nomor baris)
n_before = log["setelah panjang minimum"]
s_flag.unique(subset=["text"], keep="first").sink_parquet(STAGE)

n_dedup = pl.scan_parquet(STAGE).select(pl.len()).collect().item()
print(f"Total baris sebelum dedup : {n_before:,}")
print(f"Baris unik                : {n_dedup:,}")
print(f"Duplikat dibuang          : {n_before - n_dedup:,}")

Total baris sebelum dedup : 310,906
Baris unik                : 310,906
Duplikat dibuang          : 0


## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [6]:
BULAN = {"januari":1,"februari":2,"maret":3,"april":4,"mei":5,"juni":6,"juli":7,
         "agustus":8,"september":9,"oktober":10,"november":11,"desember":12}
NAMES = "|".join(BULAN)

# Nama yang mengandung nama lain (DKI Jakarta, Papua Barat, Kepulauan Riau) sengaja tidak dimasukkan
# agar tidak terhitung ganda.
WILAYAH = ["Aceh","Sumatera Utara","Sumatera Barat","Riau","Jambi","Sumatera Selatan","Bengkulu",
 "Lampung","Bangka Belitung","Jakarta","Jawa Barat","Jawa Tengah","Yogyakarta","Jawa Timur","Banten",
 "Bali","Nusa Tenggara Barat","Nusa Tenggara Timur","Kalimantan Barat","Kalimantan Tengah",
 "Kalimantan Selatan","Kalimantan Timur","Kalimantan Utara","Sulawesi Utara","Sulawesi Tengah",
 "Sulawesi Selatan","Sulawesi Tenggara","Gorontalo","Sulawesi Barat","Maluku","Papua"]

T = pl.col("text")
# pola tanggal: "26/8/2024", "5 Juni 2019", "FEBRUARI 01, 2017"
y_slash = T.str.extract(r"\b\d{1,2}/\d{1,2}/(\d{2,4})\b", 1).cast(pl.Int32, strict=False)
y_dmy   = T.str.extract(rf"(?i)\b\d{{1,2}}\s+(?:{NAMES})\s+((?:19|20)\d{{2}})\b", 1).cast(pl.Int32, strict=False)
y_mdy   = T.str.extract(rf"(?i)\b(?:{NAMES})\s+\d{{1,2}},?\s+((?:19|20)\d{{2}})\b", 1).cast(pl.Int32, strict=False)
m_slash = T.str.extract(r"\b\d{1,2}/(\d{1,2})/\d{2,4}\b", 1).cast(pl.Int32, strict=False)
m_dmy   = T.str.extract(rf"(?i)\b\d{{1,2}}\s+({NAMES})\s+(?:19|20)\d{{2}}\b", 1)
m_mdy   = T.str.extract(rf"(?i)\b({NAMES})\s+\d{{1,2}},?\s+(?:19|20)\d{{2}}\b", 1)

feat = (
    pl.scan_parquet(STAGE)
    .with_columns(
        T.str.count_matches(r"\S+").alias("n_words"),
        pl.coalesce(y_dmy, y_mdy, y_slash).alias("tahun"),
        pl.coalesce(
            pl.coalesce(m_dmy, m_mdy).str.to_lowercase().replace_strict(BULAN, default=None, return_dtype=pl.Int32),
            m_slash,
        ).alias("bulan"),
    )
    .with_columns(
        pl.when(pl.col("tahun") < 100).then(pl.col("tahun") + 2000).otherwise(pl.col("tahun")).alias("tahun")
    )
    .with_columns(
        pl.when(pl.col("tahun").is_between(2000, 2026)).then(pl.col("tahun")).alias("tahun"),
        pl.when(pl.col("bulan").is_between(1, 12)).then(pl.col("bulan")).alias("bulan"),
    )
    .with_columns(
        [T.str.contains(rf"(?i)\b{w}\b").alias("w_" + w.replace(" ", "_")) for w in WILAYAH]
    )
    .with_columns(
        pl.when(T.str.contains(r"(?i)\b(abstrak|abstract|metode penelitian|hasil penelitian|kata kunci)\b"))
          .then(pl.lit("jurnal"))
          .when(T.str.contains(r"(?i)\b(nih|banget|gak|nggak|sih|dong|guys|bro)\b"))
          .then(pl.lit("nonformal"))
          .otherwise(pl.lit("berita_wiki")).alias("genre")
    )
)
feat.sink_parquet(CLEAN)
print("Tersimpan:", CLEAN)

Tersimpan: ../data/processed/data_clean.parquet


In [7]:
MAX_TAHUN = 2026
val = duckdb.sql(f"""
    SELECT
        COUNT(*)                                                         AS total_baris,
        SUM(CASE WHEN text IS NULL OR text = '' THEN 1 ELSE 0 END)       AS pelanggaran_kosong,
        SUM(CASE WHEN n_chars < {MIN_CHAR} THEN 1 ELSE 0 END)            AS pelanggaran_panjang,
        COUNT(*) - COUNT(DISTINCT text)                                  AS pelanggaran_duplikat,
        SUM(CASE WHEN tahun IS NOT NULL AND (tahun < 2000 OR tahun > {MAX_TAHUN}) THEN 1 ELSE 0 END) AS tahun_invalid,
        SUM(CASE WHEN bulan IS NOT NULL AND (bulan < 1 OR bulan > 12) THEN 1 ELSE 0 END)             AS bulan_invalid,
        SUM(CASE WHEN tahun IS NULL THEN 1 ELSE 0 END)                   AS tanpa_tahun,
        SUM(CASE WHEN is_outlier_len THEN 1 ELSE 0 END)                  AS outlier_panjang
    FROM '{CLEAN}'
""").pl()
val

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

total_baris,pelanggaran_kosong,pelanggaran_panjang,pelanggaran_duplikat,tahun_invalid,bulan_invalid,tanpa_tahun,outlier_panjang
i64,"decimal[38,0]","decimal[38,0]",i64,"decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]"
310906,0,0,0,0,0,152295,6781


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [8]:
n_clean = pl.scan_parquet(CLEAN).select(pl.len()).collect().item()
print(f"Baris sebelum cleaning: {n_raw:,}")
print(f"Baris sesudah cleaning: {n_clean:,}  (dibuang {(n_raw - n_clean) / n_raw:.1%})")
print(f"Data bersih tersimpan di: {CLEAN}")
duckdb.sql(f"SUMMARIZE SELECT doc_id, n_chars, n_words, tahun, bulan, genre, is_outlier_len FROM '{CLEAN}'").show(max_width=250)

Baris sebelum cleaning: 310,930
Baris sesudah cleaning: 310,906  (dibuang 0.0%)
Data bersih tersimpan di: ../data/processed/data_clean.parquet
┌────────────────┬─────────────┬─────────────┬───────────┬───────────────┬────────────────────┬────────────────────┬─────────┬─────────┬─────────┬────────┬─────────────────┐
│  column_name   │ column_type │     min     │    max    │ approx_unique │        avg         │        std         │   q25   │   q50   │   q75   │ count  │ null_percentage │
│    varchar     │   varchar   │   varchar   │  varchar  │     int64     │      varchar       │      varchar       │ varchar │ varchar │ varchar │ int64  │  decimal(9,2)   │
├────────────────┼─────────────┼─────────────┼───────────┼───────────────┼────────────────────┼────────────────────┼─────────┼─────────┼─────────┼────────┼─────────────────┤
│ doc_id         │ BIGINT      │ 1           │ 310930    │        301583 │ 155464.4841881469  │ 89761.23881690712  │ 77826   │ 155481  │ 233074  │ 310906 │      

## Catatan Milestone 2

Dokumentasikan di sini: dimensi kualitas data yang paling bermasalah, strategi yang dipilih untuk missing values dan outlier, jumlah baris yang berubah, serta aturan validasi yang diterapkan.